# Data Mining Assignment — Apache Spark Data Lake Preprocessing

**Dataset:** `sales.csv`  |  **Environment:** Google Colab + PySpark

This notebook builds a simple **Data Lake** (Bronze → Silver) for the sales dataset:

| Layer | Content |
|-------|---------|
| **Bronze** | Raw `sales.csv`, kept byte-for-byte unchanged |
| **Silver** | Basic preprocessing done with Apache Spark (no Gold layer, no detailed analysis) |

**How to run:** `Runtime → Run all`. When the upload prompt appears in Section 1, choose `sales.csv`.
All discrepancy counts below are produced by Spark operations — nothing is hard-coded.

## 1. Data Lake Structure

Install PySpark, then create the folders:

```
data_lake/
|-- bronze/
|   `-- sales.csv
`-- silver/
    `-- preprocessed_sales/
```

In [ ]:
!pip install -q pyspark

: 

In [ ]:
import os, re, glob, shutil, hashlib

BASE       = "data_lake"
BRONZE     = f"{BASE}/bronze"
SILVER     = f"{BASE}/silver"
SILVER_OUT = f"{SILVER}/preprocessed_sales"      # Spark output folder
BRONZE_FILE = f"{BRONZE}/sales.csv"

for d in (BRONZE, SILVER_OUT):
    os.makedirs(d, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# Put the provided sales.csv into Bronze (plain byte-for-byte copy, no edits).
if not os.path.exists(BRONZE_FILE):
    source = next((p for p in ("sales.csv", "/content/sales.csv") if os.path.exists(p)), None)
    if source is None:
        try:
            from google.colab import files          # only available inside Colab
            uploaded = files.upload()               # <-- choose sales.csv here
            source = "sales.csv" if "sales.csv" in uploaded else next(iter(uploaded))
        except ImportError:
            raise FileNotFoundError("sales.csv not found. Place it next to this notebook and re-run.")
    shutil.copyfile(source, BRONZE_FILE)

BRONZE_SHA256 = sha256(BRONZE_FILE)                 # used at the end to prove Bronze is unchanged
print("Bronze file :", BRONZE_FILE)
print("SHA-256     :", BRONZE_SHA256)
for root, dirs, fs in os.walk(BASE):
    print(" ", root, fs)

FileNotFoundError: sales.csv not found. Place it next to this notebook and re-run.

## 2. Load and Inspect with Spark

The file is read with `inferSchema=False`, so **every column is loaded as a string**. This is deliberate: it lets us
see the raw problems (e.g. `"abc"` in a numeric column, or mixed date formats) instead of Spark silently turning them into nulls.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("DataLake_Sales_Preprocessing")
         .master("local[*]")
         .config("spark.sql.ansi.enabled", "false")                 # bad casts / dates -> null instead of errors
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")  # strict, predictable date parsing
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

df_raw = (spark.read
          .option("header", True)
          .option("inferSchema", False)
          .option("multiLine", True)
          .option("escape", '"')
          .csv(BRONZE_FILE))

raw_count = df_raw.count()

print("=== Schema ===")
df_raw.printSchema()
print("=== Sample rows ===")
df_raw.show(10, truncate=False)
print(f"=== Total record count (raw) === {raw_count}")

### 2.1 Identify column roles

Columns are classified automatically from their names / values (quantity, unit price, discount, dates, IDs, other numeric, text).
The detected roles are printed — **if one is wrong, edit the lists in this cell and re-run.**

In [ ]:
ALL_COLS = df_raw.columns
PLACEHOLDERS = ["", "na", "n/a", "nan", "null", "none", "nil", "-", "--", "?", "missing"]   # strings that really mean "missing"

# ---- generic helpers -------------------------------------------------------
def squash(c):
    """Collapse any run of whitespace (incl. non-breaking space) to one space and trim the ends."""
    return F.trim(F.regexp_replace(F.col(c) if isinstance(c, str) else c, r"[\s\u00A0]+", " "))

def is_missing(c):
    """True for real NULLs and for placeholder strings such as '', 'NA', 'N/A', 'null'."""
    return F.col(c).isNull() | F.lower(squash(c)).isin(PLACEHOLDERS)

def to_num(c, strip_pct=False):
    """Clean a numeric string (commas, currency symbols, optional %) and cast to double; unparseable -> NULL."""
    s = squash(c)
    s = F.regexp_replace(s, r"(?i)^(rs\.?|inr)", "")
    s = F.regexp_replace(s, r"[,\s₹$€£]", "")
    if strip_pct:
        s = F.regexp_replace(s, r"%$", "")
    v = s.cast("double")
    return F.when(v.between(-1e15, 1e15), v)          # also removes NaN / Infinity

# ---- role detection --------------------------------------------------------
def match(pattern, exclude=()):
    return [c for c in ALL_COLS if re.search(pattern, c, re.I) and c not in exclude]

DATE_COLS     = match(r"date|(^|_)dt($|_)|timestamp")
DISCOUNT_COLS = match(r"disc")
QTY_COLS      = match(r"qty|quantit|^units?$|units?.?(sold|ordered)", exclude=DISCOUNT_COLS)
_not_unit     = r"total|amount|revenue|sales|net|gross|profit|cost"
PRICE_COLS    = (match(r"unit.?price|^price$|price.?per|^rate$|^mrp$", exclude=DISCOUNT_COLS + QTY_COLS)
                 or [c for c in match(r"price|rate|mrp", exclude=DISCOUNT_COLS + QTY_COLS) if not re.search(_not_unit, c, re.I)])
ID_COLS       = match(r"(^|[^a-z])(id|code|sku|uuid|no|number)([^a-z]|$)|id$")
EMAIL_COLS    = match(r"mail", exclude=ID_COLS)

_assigned = set(DATE_COLS + DISCOUNT_COLS + QTY_COLS + PRICE_COLS + ID_COLS + EMAIL_COLS)
_rest = [c for c in ALL_COLS if c not in _assigned]

# remaining columns whose (non-missing) values are >= 90% numeric are treated as "other numeric"
if _rest:
    _stats = df_raw.agg(*[
        F.count(F.when(~is_missing(c), 1)).alias(f"p_{i}") for i, c in enumerate(_rest)
    ] + [
        F.count(F.when(~is_missing(c) & to_num(c).isNotNull(), 1)).alias(f"n_{i}") for i, c in enumerate(_rest)
    ]).first().asDict()
else:
    _stats = {}
OTHER_NUM_COLS = [c for i, c in enumerate(_rest) if _stats[f"p_{i}"] > 0 and _stats[f"n_{i}"] / _stats[f"p_{i}"] >= 0.9]
TEXT_COLS = [c for c in _rest if c not in OTHER_NUM_COLS]
PRODUCT_COL = next((c for c in TEXT_COLS if re.search(r"product|item", c, re.I)), None)

print("Quantity columns      :", QTY_COLS)
print("Unit-price columns    :", PRICE_COLS)
print("Discount columns      :", DISCOUNT_COLS)
print("Date columns          :", DATE_COLS)
print("ID columns            :", ID_COLS)
print("E-mail columns        :", EMAIL_COLS)
print("Other numeric columns :", OTHER_NUM_COLS)
print("Text columns          :", TEXT_COLS)
print("Product column        :", PRODUCT_COL)

# ---- date parsing helpers --------------------------------------------------
DATE_FORMATS = ["yyyy-MM-dd", "yyyy/MM/dd", "d/M/yyyy", "d-M-yyyy", "d.M.yyyy", "dd-MMM-yyyy", "d MMM yyyy",
                "d MMMM yyyy", "MMM d, yyyy", "MMMM d, yyyy", "yyyyMMdd", "yyyy-MM-dd HH:mm:ss",
                "yyyy-MM-dd'T'HH:mm:ss", "M/d/yyyy"]       # d/M tried before M/d (day-first); M/d only rescues e.g. 05/13/2024
MIN_DATE = "2000-01-01"                                   # earliest date accepted as plausible; latest = today

def parse_date_any(c):
    s = squash(c)
    return F.coalesce(*[F.to_date(s, f) for f in DATE_FORMATS])

def parse_date(c):
    d = parse_date_any(c)
    return F.when(d.between(F.lit(MIN_DATE).cast("date"), F.current_date()), d)

### 2.2 Duplicates

In [ ]:
distinct_count = df_raw.distinct().count()
dup_rows = raw_count - distinct_count
print(f"Total rows: {raw_count} | Distinct rows: {distinct_count} | Exact duplicate rows: {dup_rows}")

print("Rows that appear more than once:")
(df_raw.groupBy(*df_raw.columns).count()
       .filter(F.col("count") > 1)
       .show(truncate=False))

### 2.3 Null / missing values

Counts both real NULLs and placeholder strings (`""`, `NA`, `N/A`, `null`, `-`, ...).

In [ ]:
miss = df_raw.agg(*[F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in ALL_COLS]).first().asDict()
plac = df_raw.agg(*[F.count(F.when(F.col(c).isNotNull() & is_missing(c), 1)).alias(c) for c in ALL_COLS]).first().asDict()

print(f"{'column':<25}{'true NULLs':>12}{'placeholder strings':>22}{'total missing':>16}")
for c in ALL_COLS:
    print(f"{c:<25}{miss[c]:>12}{plac[c]:>22}{miss[c] + plac[c]:>16}")

### 2.4 Text-format inconsistencies

Leading/trailing spaces, repeated inner spaces and capitalisation variants (e.g. `electronics`, `ELECTRONICS`, `Electronics`).

In [ ]:
report = []
for c in TEXT_COLS:
    raw = F.col(c)
    r = df_raw.agg(
        F.count(F.when(raw.isNotNull() & (raw != F.trim(raw)), 1)).alias("edge_sp"),
        F.count(F.when(raw.rlike(r"\s{2,}"), 1)).alias("inner_sp"),
        F.countDistinct(raw).alias("d_raw"),
        F.countDistinct(F.initcap(squash(c))).alias("d_clean"),
    ).first()
    report.append((c, r["edge_sp"], r["inner_sp"], r["d_raw"], r["d_clean"]))

print("Text columns summary:")
spark.createDataFrame(report, "column string, leading_trailing_spaces long, repeated_inner_spaces long, "
                              "distinct_raw_values long, distinct_after_cleaning long").show(truncate=False)

for c in TEXT_COLS:
    print(f"--- spelling / case variants in '{c}' (same value written in several ways) ---")
    (df_raw.filter(F.col(c).isNotNull())
           .groupBy(F.lower(squash(c)).alias("normalised"))
           .agg(F.collect_set(F.col(c)).alias("variants_found"), F.countDistinct(F.col(c)).alias("n_variants"))
           .filter(F.col("n_variants") > 1)
           .show(10, truncate=False))

### 2.5 Invalid numerical values

Validation rules used: **quantity** must be a positive whole number, **unit price** must be > 0, **discount %** must be between 0 and 100.

In [ ]:
def num_report(cols, label, rules):
    for c in cols:
        v = to_num(c, strip_pct=(label == "discount"))
        present = ~is_missing(c)
        aggs = [F.count(F.when(present & v.isNull(), 1)).alias("non_numeric_text")]
        for name, cond in rules(v):
            aggs.append(F.count(F.when(cond, 1)).alias(name))
        aggs += [F.min(v).alias("min_value"), F.max(v).alias("max_value")]
        print(f"[{label}] column '{c}':")
        for k, val in df_raw.agg(*aggs).first().asDict().items():
            print(f"    {k:<22}: {val}")

num_report(QTY_COLS,      "quantity",   lambda v: [("zero_or_negative", v <= 0), ("non_integer", v.isNotNull() & (v != F.floor(v)))])
num_report(PRICE_COLS,    "unit price", lambda v: [("zero_or_negative", v <= 0)])
num_report(DISCOUNT_COLS, "discount",   lambda v: [("below_0", v < 0), ("above_100", v > 100)])

### 2.6 Inconsistent / invalid dates

First, which formats are present; then how many values cannot be parsed at all (e.g. `2024-02-30`, free text) or fall outside the plausible range (`MIN_DATE` … today).

In [ ]:
for c in DATE_COLS:
    s = squash(c)
    shape = (F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}$"), "yyyy-MM-dd")
              .when(s.rlike(r"^\d{4}/\d{2}/\d{2}$"), "yyyy/MM/dd")
              .when(s.rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"), "dd/MM/yyyy or MM/dd/yyyy")
              .when(s.rlike(r"^\d{1,2}-\d{1,2}-\d{4}$"), "dd-MM-yyyy")
              .when(s.rlike(r"^\d{1,2}-[A-Za-z]{3}-\d{4}$"), "dd-MMM-yyyy")
              .when(s.rlike(r"^\d{8}$"), "yyyyMMdd")
              .when(s.rlike(r"^[A-Za-z]+ \d{1,2}, \d{4}$"), "MMMM d, yyyy")
              .otherwise("other / free text"))
    print(f"[dates] column '{c}' — formats found:")
    (df_raw.filter(~is_missing(c)).groupBy(shape.alias("format_pattern")).count().orderBy(F.desc("count")).show(truncate=False))

    r = df_raw.agg(
        F.count(F.when(is_missing(c), 1)).alias("missing"),
        F.count(F.when(~is_missing(c) & parse_date_any(c).isNull(), 1)).alias("unparseable"),
        F.count(F.when(parse_date_any(c).isNotNull() & parse_date(c).isNull(), 1)).alias("parsed_but_out_of_range"),
    ).first().asDict()
    print(f"    {r}\n")

## 3. Basic Preprocessing — Silver Layer

| # | Operation | Method |
|---|-----------|--------|
| 1 | Exact duplicates | Keep the first occurrence of rows identical in **all** columns |
| 2 | Missing values | Placeholder strings (`NA`, `N/A`, blank …) → NULL; text → `"Unknown"`; discount → `0`; quantity → median; unit price → median of the same product (else overall median); dates stay NULL (a date must not be invented) |
| 3 | Text fields | Trim, collapse repeated spaces, `Title Case` (e-mails → lower case; ID columns only trimmed) |
| 4 | Validation | Quantity must be a positive integer, unit price > 0, discount in [0, 100]. Invalid values are set to NULL and then treated like missing values — **the record itself is kept** |
| 5 | Dates | Parsed from the formats found above and written as `yyyy-MM-dd`; unparseable / implausible dates → NULL |
| 6 | Preserve records | Only exact duplicates are dropped; unusual-but-valid values (e.g. very large quantities) are left alone |

In [ ]:
ops_log = {}        # every number reported at the end comes from a Spark action below

# -- Step 1: remove exact duplicate rows (keeps first occurrence and original row order) ----------------
df_idx = df_raw.withColumn("_rid", F.monotonically_increasing_id())
df1 = (df_idx.groupBy(*ALL_COLS).agg(F.min("_rid").alias("_rid")).orderBy("_rid"))
after_dedup = df1.count()
ops_log["Exact duplicate rows removed"] = raw_count - after_dedup

# -- Step 2a: placeholder strings -> real NULLs ---------------------------------------------------------
ops_log["Placeholder strings (NA, N/A, blank, ...) converted to NULL"] = sum(
    df1.agg(*[F.count(F.when(F.col(c).isNotNull() & is_missing(c), 1)) for c in ALL_COLS]).first())
df2 = df1.select("_rid", *[F.when(is_missing(c), F.lit(None)).otherwise(F.col(c)).alias(c) for c in ALL_COLS])

# -- Step 3: text standardisation -----------------------------------------------------------------------
std = {}
for c in TEXT_COLS:
    std[c] = F.initcap(squash(c))
for c in ID_COLS:
    std[c] = squash(c)
for c in EMAIL_COLS:
    std[c] = F.lower(squash(c))

changed = df2.agg(*[F.count(F.when(F.col(c).isNotNull() & (F.col(c) != std[c]), 1)) for c in std]).first() if std else []
ops_log["Text cells trimmed / re-spaced / re-capitalised"] = sum(changed)
ops_log["Missing text values filled with 'Unknown'"] = sum(
    df2.agg(*[F.count(F.when(F.col(c).isNull(), 1)) for c in TEXT_COLS]).first()) if TEXT_COLS else 0

df3 = df2
for c, expr in std.items():
    expr = F.when(F.col(c).isNull(), F.lit("Unknown") if c in TEXT_COLS else F.lit(None)).otherwise(expr)
    df3 = df3.withColumn(c, expr)

# -- Step 4: numeric validation + missing-value handling --------------------------------------------------
def validate(df, cols, valid_fn, label, strip_pct=False):
    """Return df with invalid values turned into NULL; log how many were missing / invalid."""
    for c in cols:
        v = to_num(c, strip_pct)
        valid = valid_fn(v)
        n = df.agg(F.count(F.when(F.col(c).isNull(), 1)).alias("missing"),
                   F.count(F.when(F.col(c).isNotNull() & ~valid, 1)).alias("invalid")).first()
        ops_log[f"{label} '{c}': missing values"] = n["missing"]
        ops_log[f"{label} '{c}': invalid values -> NULL"] = n["invalid"]
        df = df.withColumn(c, F.when(valid, v))
    return df

df4 = validate(df3, QTY_COLS,      lambda v: v.isNotNull() & (v > 0) & (v == F.floor(v)), "Quantity")
df4 = validate(df4, PRICE_COLS,    lambda v: v.isNotNull() & (v > 0),                      "Unit price")
df4 = validate(df4, DISCOUNT_COLS, lambda v: v.isNotNull() & (v >= 0) & (v <= 100),        "Discount %", strip_pct=True)

# other numeric columns (e.g. totals): just clean + cast, no imputation
for c in OTHER_NUM_COLS:
    df4 = df4.withColumn(c, to_num(c))

# imputation (NULLs created by missing OR invalid values)
for c in QTY_COLS:
    med = df4.approxQuantile(c, [0.5], 0.0)
    fill = float(round(med[0])) if med else 1.0
    ops_log[f"Quantity '{c}': NULLs filled with median"] = df4.filter(F.col(c).isNull()).count()
    df4 = df4.withColumn(c, F.coalesce(F.col(c), F.lit(fill)).cast("int"))

for c in PRICE_COLS:
    med = df4.approxQuantile(c, [0.5], 0.0)
    gmed = float(med[0]) if med else 0.0
    ops_log[f"Unit price '{c}': NULLs filled (product median, else overall median)"] = df4.filter(F.col(c).isNull()).count()
    if PRODUCT_COL:
        pm = df4.groupBy(PRODUCT_COL).agg(F.percentile_approx(c, 0.5).alias("_pmed"))
        df4 = df4.join(pm, on=PRODUCT_COL, how="left")
        df4 = df4.withColumn(c, F.coalesce(F.col(c), F.col("_pmed"), F.lit(gmed))).drop("_pmed")
    else:
        df4 = df4.withColumn(c, F.coalesce(F.col(c), F.lit(gmed)))
    df4 = df4.withColumn(c, F.round(F.col(c), 2))

for c in DISCOUNT_COLS:
    ops_log[f"Discount '{c}': NULLs filled with 0"] = df4.filter(F.col(c).isNull()).count()
    df4 = df4.withColumn(c, F.round(F.coalesce(F.col(c), F.lit(0.0)), 2))

# -- Step 5: dates -> one consistent format (yyyy-MM-dd) ---------------------------------------------------
df5 = df4
for c in DATE_COLS:
    n = df5.agg(F.count(F.when(F.col(c).isNull(), 1)).alias("missing"),
                F.count(F.when(F.col(c).isNotNull() & parse_date(c).isNull(), 1)).alias("invalid")).first()
    ops_log[f"Date '{c}': missing values (left NULL)"] = n["missing"]
    ops_log[f"Date '{c}': invalid / unparseable / out-of-range -> NULL"] = n["invalid"]
    df5 = df5.withColumn(c, F.date_format(parse_date(c), "yyyy-MM-dd"))

# -- Step 6: final Silver DataFrame (original column order & row order) ------------------------------------
df_silver = df5.orderBy("_rid").select(*ALL_COLS)
df_silver = df_silver.cache()
silver_count = df_silver.count()
print(f"Silver DataFrame ready: {silver_count} rows x {len(df_silver.columns)} columns")
df_silver.printSchema()
df_silver.show(10, truncate=False)

### Save to the Silver layer

Spark writes a folder of part files; with `coalesce(1)` there is exactly one part file, which is also copied to
`data_lake/silver/sales_silver.csv` (the file required for submission).

In [ ]:
(df_silver.coalesce(1)
          .write.mode("overwrite")
          .option("header", True)
          .csv(SILVER_OUT))

part_file = glob.glob(f"{SILVER_OUT}/part-*.csv")[0]
SILVER_SINGLE = f"{SILVER}/sales_silver.csv"
shutil.copyfile(part_file, SILVER_SINGLE)

print("Spark output folder :", SILVER_OUT, "->", sorted(os.listdir(SILVER_OUT)))
print("Single CSV copy     :", SILVER_SINGLE)

## 4. Verification

In [ ]:
silver_back = (spark.read.option("header", True).option("inferSchema", True).csv(SILVER_OUT))

print("=== Row count before / after preprocessing ===")
print(f"Before (Bronze, raw) : {raw_count}")
print(f"After  (Silver)      : {silver_back.count()}")
print(f"Rows removed         : {raw_count - silver_back.count()}")

print("\n=== Silver data read back with Spark ===")
silver_back.printSchema()
silver_back.show(10, truncate=False)

In [ ]:
# Sanity checks on the Silver data, computed with Spark
checks = {}
checks["exact duplicate rows left"] = silver_back.count() - silver_back.distinct().count()
for c in QTY_COLS:
    checks[f"'{c}' <= 0 or NULL"] = silver_back.filter(F.col(c).isNull() | (F.col(c) <= 0)).count()
for c in PRICE_COLS:
    checks[f"'{c}' <= 0 or NULL"] = silver_back.filter(F.col(c).isNull() | (F.col(c) <= 0)).count()
for c in DISCOUNT_COLS:
    checks[f"'{c}' outside [0,100] or NULL"] = silver_back.filter(F.col(c).isNull() | (F.col(c) < 0) | (F.col(c) > 100)).count()
for c in DATE_COLS:
    checks[f"'{c}' not in yyyy-MM-dd (non-NULL)"] = silver_back.filter(
        F.col(c).isNotNull() & ~F.col(c).cast("string").rlike(r"^\d{4}-\d{2}-\d{2}$")).count()
    checks[f"'{c}' NULL (unparseable/missing, kept as NULL)"] = silver_back.filter(F.col(c).isNull()).count()
for c in TEXT_COLS:
    checks[f"'{c}' with leading/trailing/double spaces"] = silver_back.filter(
        F.col(c).cast("string").rlike(r"^\s|\s$|\s{2,}")).count()
for k, v in checks.items():
    print(f"{k:<60}: {v}")

print("\nBronze file unchanged:", sha256(BRONZE_FILE) == BRONZE_SHA256)

### Preprocessing operations performed (summary)

In [ ]:
print(f"Rows: {raw_count} (Bronze)  ->  {silver_back.count()} (Silver)\n")
print("Operations performed (all counts obtained from Spark):")
for k, v in ops_log.items():
    print(f"  - {k}: {v}")
print("\nOther operations: date formats unified to yyyy-MM-dd; Silver saved as CSV in data_lake/silver/preprocessed_sales/ "
      "(single copy at data_lake/silver/sales_silver.csv); Bronze layer untouched.")

### Download the file needed for the pull-request (Colab only)

In [ ]:
try:
    from google.colab import files
    files.download(SILVER_SINGLE)          # silver/sales_silver.csv  -> upload this to GitHub
except ImportError:
    print("Not running in Colab - the file is at", SILVER_SINGLE)